# Комментарий

Скопируй тестовый набор данных

На отдельной вкладке внутри твоей копии собери ссылки на файлы для проверки заданий №2 и №3. Итого в твоей таблице должны быть следующие вкладки: 
1. Data
2. Сводная таблица из Data
3. Вкладка с ссылками на визуализацию и ссылкой на Jupyter Notebook

Не забудь расшерить доступ по ссылке с возможностью оставить комментарий 

In [1]:
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

import scipy
from scipy import stats

for i in (np, pd, matplotlib, scipy):
    print(i.__version__)

2.3.3
2.3.3
3.10.8
1.16.3


In [2]:
df = pd.read_csv("./Data для тестового - Data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Название рассылки       218 non-null    object
 1   Название кампании       218 non-null    object
 2   Направление             218 non-null    object
 3   Месяц                   218 non-null    object
 4   Дата                    218 non-null    object
 5   Год                     218 non-null    int64 
 6   Номер недели            218 non-null    int64 
 7   День недели             218 non-null    int64 
 8   День недели.1           218 non-null    object
 9   Время                   218 non-null    object
 10  Веб-версия              218 non-null    object
 11  Тема письма             218 non-null    object
 12  Сегмент                 218 non-null    object
 13  Отправлено              218 non-null    object
 14  Доставлено              218 non-null    object
 15  Открыт

In [3]:
def extract_digit(data):
    if data.dtype in ("int", "float"):
        return data

    return data.str.replace('\xa0', '').astype("int64")


cols2digit = ["Отправлено", "Доставлено", "Открытия", "Клики", "Отписки"]
for title in cols2digit: 
    df[title] = extract_digit(df[title])

df["Дата"] = pd.to_datetime(df["Дата"])

C:\Users\Егор\AppData\Local\Temp\ipykernel_1168\219293355.py:12: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Дата"] = pd.to_datetime(df["Дата"])


# Задание №1

Сделай сводную таблицу для набора данных из этого файла, которая отразит динамику основных метрик для email-рассылок (Delivery rate, Open rate, CTOR, UR).

In [4]:
pivot_table = df.groupby(["Дата"]).agg({title: "sum" for title in cols2digit})

pivot_table["Deliver rate"] = pivot_table["Доставлено"] / pivot_table["Отправлено"]
pivot_table["Open rate"] = pivot_table["Открытия"] / pivot_table["Доставлено"]
pivot_table["CTOR"] = pivot_table["Клики"] / pivot_table["Открытия"]
pivot_table["UR"] = pivot_table["Отписки"] / pivot_table["Доставлено"]

# pivot_table = pivot_table.drop(cols2digit, axis=1)

pivot_table.to_excel("./pivot_table.xlsx")
pivot_table.to_csv("./pivot_table.csv")

pivot_table

,Отправлено,Доставлено,Открытия,Клики,Отписки,Deliver rate,Open rate,CTOR,UR
Дата,,,,,,,,,
2021-04-15,688566,654138,94261,3676,4514,0.950000,0.144100,0.038998,0.006901
2021-04-21,627527,596151,83342,6001,4113,0.950001,0.139800,0.072005,0.006899
2021-04-22,1886230,1791919,285811,34297,12364,0.950000,0.159500,0.119999,0.006900
2021-04-23,2342323,2225207,363154,32684,15354,0.950000,0.163200,0.090000,0.006900
2021-04-30,724212,688001,99141,8328,4747,0.949999,0.144100,0.084002,0.006900
...,...,...,...,...,...,...,...,...,...
2022-04-26,1771599,1683019,235623,17466,13699,0.950000,0.140000,0.074127,0.008140
2022-05-05,1129965,1073467,150285,10821,7407,0.950000,0.140000,0.072003,0.006900
2022-05-12,2378647,2259715,316360,37963,15592,0.950000,0.140000,0.119999,0.006900


# Задание №2

Построй визуализацию для набора данных из задания №1 (инструменты: DataLens или Looker Data Studio). Элементы, которые необходимо отразить в дашборде:
Динамика ключевых метрик для оценки эффективности рассылок 
Визуализируй воронку продаж

[Дашборд - ключевые метрики](https://datalens.ru/voa5xz5rk5fqf-klyuchevye-metriki)

# Задание №3

На основе уже знакомого тебе тестового набора данных из первого задания посчитай следующие метрики в тетрадке Jupyter Notebook:
- Delivery rate
- Open rate
- Click to Open rate
- Unsubscribe rate
- Выяви лучшую тему

In [5]:
def cr(df, a, b):
    return df[a].sum() / df[b].sum()

deliver_rate = cr(df, "Доставлено", "Отправлено")
open_rate = cr(df, "Открытия", "Доставлено")
ctor = cr(df, "Клики", "Открытия")
ur = cr(df, "Отписки", "Доставлено")

print(f"""
Deliver rate = {deliver_rate:.2%}
Open rate = {open_rate:.2%}
CTOR = {ctor:.2%}
UR = {ur:.2%}
""")

subjects = df.groupby(["Тема письма "], as_index=False).agg({title: "sum" for title in ["Доставлено", "Открытия"]})
subjects["Open rate"] = subjects["Открытия"] / subjects["Доставлено"]

best_subjects = subjects[subjects["Open rate"].round(4) == subjects["Open rate"].max()]

print("Лучшие темы письма:")
for (best_subject, delivered, opened, open_rate) in best_subjects.values:
    print(f"{best_subject}\t Доставлено = {delivered} | Открыто = {opened} | Open rate {open_rate:.2%}")


Deliver rate = 97.70%
Open rate = 13.62%
CTOR = 8.10%
UR = 2.62%

Лучшие темы письма:
Тема письма 1	 Доставлено = 741750 | Открыто = 148350 | Open rate 20.00%
Тема письма 101	 Доставлено = 1324136 | Открыто = 264827 | Open rate 20.00%
